In [1]:
import re
import json
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration

c:\Users\acaru\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Loading the trained model
model_path = "../resume-builder/t5_resume_model_no_optimizer"
tokenizer = T5Tokenizer.from_pretrained(model_path)
model = T5ForConditionalGeneration.from_pretrained(model_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Model loaded successfully.")

Model loaded successfully.


In [4]:
def extract_json_from_text(text):
    """ Extract JSON content from model output using regex. """
    json_pattern = r"\{.*?\}"  # Match content inside `{...}`
    match = re.search(json_pattern, text, re.DOTALL)

    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            return {"error": "Extracted JSON is invalid"}
    return {"error": "No JSON found in model output"}

In [5]:
def generate_resume(minimal_input):
    input_text = f"""
    Convert the following structured data into valid JSON format:
    
    Name: {minimal_input['fullName']}
    Job Title: {minimal_input['jobTitle']}
    Contact: {minimal_input['contact']}
    Location: {minimal_input['location']}
    Email: {minimal_input['email']}
    
    Experience:
    {" | ".join([f"{exp['role']} at {exp['company']} ({exp['duration']})" for exp in minimal_input['experience']])}
    
    Education:
    {" | ".join([f"{edu['degree']} from {edu['institution']} ({edu['year_end']})" for edu in minimal_input['education']])}
    
    Skills:
    {", ".join(minimal_input['skills'])}
    
    Output JSON format:
    {{
      "fullName": "John Doe",
      "jobTitle": "Software Engineer",
      "contact": "123-456-7890",
      "location": "New York, USA",
      "email": "johndoe@example.com",
      "experience": [
          {{"company": "Google", "role": "Software Developer", "duration": "2 years"}},
          {{"company": "Amazon", "role": "Backend Engineer", "duration": "3 years"}}
      ],
      "education": [
          {{"institution": "MIT", "degree": "B.Tech in CS", "year_end": "2022"}}
      ],
      "skills": ["Python", "Machine Learning", "Cloud Computing"]
    }}

    Provide only the JSON output without any extra text.
    """

    # Tokenize input
    input_ids = tokenizer(input_text, return_tensors="pt", padding="max_length", truncation=True, max_length=512).input_ids.to(device)

    # Generate output
    with torch.no_grad():
        output_ids = model.generate(input_ids, max_length=512, num_return_sequences=1)

    # Decode output
    output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    print("\n🔹 Raw Model Output:\n", output_text)  # Debugging Output

    # Extract JSON
    generated_resume = extract_json_from_text(output_text)

    return generated_resume

In [6]:
# Classification of skills
def classify_skills(resume_json):
    tech_skills = ["Python", "Machine Learning", "Cloud Computing", "Java", "C++", "TensorFlow"]
    soft_skills = ["Leadership", "Communication", "Problem Solving", "Teamwork"]

    classified_skills = {"Technical Skills": [], "Soft Skills": []}

    for skill in resume_json.get("skills", []):
        if skill in tech_skills:
            classified_skills["Technical Skills"].append(skill)
        elif skill in soft_skills:
            classified_skills["Soft Skills"].append(skill)
    
    resume_json["skills"] = classified_skills
    return resume_json

In [7]:
# Example usage

minimal_input = {
    "fullName": "John Doe",
    "jobTitle": "Software Engineer",
    "contact": "123-456-7890",
    "location": "New York, USA",
    "email": "johndoe@example.com",
    "experience": [
        {"company": "Google", "role": "Software Developer", "place": "NY", "duration": "2 years"},
        {"company": "Amazon", "role": "Backend Engineer", "place": "Seattle", "duration": "3 years"}
    ],
    "education": [
        {"institution": "MIT", "degree": "B.Tech in CS", "year_end": "2022"}
    ],
    "skills": ["Python", "Machine Learning", "Cloud Computing"]
}

generated_resume = generate_resume(minimal_input)
print("Generated Resume JSON:", json.dumps(generated_resume, indent=4))

classified_resume = classify_skills(generated_resume)
print("Classified Resume JSON:", json.dumps(classified_resume, indent=4))


🔹 Raw Model Output:
 "A": "A": "A": "A": "A": "A": "A": "A": "A": "A": "A": "A": "A": "A": "A": "A": "A": "A": "A": "A": "A": "A": "A": "A": "A": "A": "A": "A": "
Generated Resume JSON: {
    "error": "No JSON found in model output"
}
Classified Resume JSON: {
    "error": "No JSON found in model output",
    "skills": {
        "Technical Skills": [],
        "Soft Skills": []
    }
}


In [1]:
import json
from transformers import T5ForConditionalGeneration, T5Tokenizer

# Load trained model
model_path = "./t5_resume_model_no_optimizer"
model = T5ForConditionalGeneration.from_pretrained(model_path)
tokenizer = T5Tokenizer.from_pretrained(model_path)

# Sample minimal input
minimal_input = {
    "fullName": "John Doe",
    "jobTitle": "Software Engineer",
    "contact": "123-456-7890",
    "location": "New York, USA",
    "email": "johndoe@example.com",
    "experience": [
        {"company": "Google", "role": "Software Developer", "place": "NY", "duration": "2 years"},
        {"company": "Amazon", "role": "Backend Engineer", "place": "Seattle", "duration": "3 years"}
    ],
    "education": [
        {"institution": "MIT", "degree": "B.Tech in CS", "year_end": "2022"}
    ],
    "skills": ["Python", "Machine Learning", "Cloud Computing"]
}

    


c:\Users\acaru\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Generate resume function
def generate_resume(minimal_input):
  input_text = f"""
  Give the missing fields and convert the following structured data into valid JSON format:
  
  Name: {minimal_input['fullName']}
  Job Title: {minimal_input['jobTitle']}
  Contact: {minimal_input['contact']}
  Location: {minimal_input['location']}
  Email: {minimal_input['email']}
  
  Experience:
  {" | ".join([f"{exp['role']} at {exp['company']} ({exp['duration']})" for exp in minimal_input['experience']])}
  
  Education:
  {" | ".join([f"{edu['degree']} from {edu['institution']} ({edu['year_end']})" for edu in minimal_input['education']])}
  
  Skills:
  {", ".join(minimal_input['skills'])}
  
  Output JSON format:
  {{
    "fullName": "John Doe",
    "jobTitle": "Software Engineer",
    "contact": "123-456-7890",
    "location": "New York, USA",
    "email": "johndoe@example.com",
    "experience": [
      {{"company": "Google", "role": "Software Developer", "duration": "2 years", "description": "Worked on XYZ project"}},
      {{"company": "Amazon", "role": "Backend Engineer", "duration": "3 years"}}
    ],
    "education": [
      {{"institution": "MIT", "degree": "B.Tech in CS", "year_end": "2022"}}
    ],
    "skills": ["Python", "Machine Learning", "Cloud Computing"]
  }}

  Provide only the JSON output without any extra text.
  """

  # Tokenize input
  input_ids = tokenizer(input_text, return_tensors="pt", padding="max_length", truncation=True, max_length=512).input_ids.to(model.device)

  # Generate output
  with torch.no_grad():
    output_ids = model.generate(input_ids, max_length=512, num_return_sequences=1)

  # Decode output
  output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

  # Extract JSON
  generated_resume = extract_json_from_text(output_text)

  return generated_resume